In [1]:
!pip install -q -U transformers datasets accelerate

In [2]:
import pandas as pd
import torch
import json
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
from google.colab import files
uploaded = files.upload()

Saving DEV_TEST_Merged_ASQE_N2500.jsonl to DEV_TEST_Merged_ASQE_N2500 (1).jsonl
Saving Train_validation_Merged_ASQE_N4000.jsonl to Train_validation_Merged_ASQE_N4000 (1).jsonl


In [4]:
def load_jsonl(file_path):
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

train_df = load_jsonl("Train_validation_Merged_ASQE_N4000.jsonl")
test_df = load_jsonl("DEV_TEST_Merged_ASQE_N2500.jsonl")

print(train_df.columns)
train_df.head()

Index(['text', 'labels'], dtype='object')


,text,labels
0,He is not helpful at all and makes fun of stud...,"[{'aspect': 'He', 'opinion': 'Very rude', 'pol..."
1,"This course was amazing. For the first time, t...","[{'aspect': 'This course', 'opinion': 'I loved..."
2,Exams were very easy and he allowed us to do e...,"[{'aspect': 'Exams', 'opinion': 'very easy', '..."
3,The lecturers from Exeter have helped me throu...,"[{'aspect': 'lecturers', 'opinion': 'helped me..."
4,"Extremely boring and lectures were useless, bu...","[{'aspect': 'lectures', 'opinion': 'Extremely ..."


In [5]:
def convert_to_absa(df):
    rows = []

    for _, row in df.iterrows():
        sentence = row["text"]
        label_list = row["labels"]

        for item in label_list:
            aspect = item.get("aspect")
            sentiment = item.get("polarity")

            if aspect is None or aspect == "null":
                continue

            rows.append({
                "sentence": sentence,
                "aspect": aspect,
                "sentiment": sentiment
            })

    df_new = pd.DataFrame(rows)
    df_new = df_new.drop_duplicates().reset_index(drop=True)

    return df_new

train_absa = convert_to_absa(train_df)
test_absa = convert_to_absa(test_df)

train_absa["sentiment"] = train_absa["sentiment"].str.capitalize()
test_absa["sentiment"] = test_absa["sentiment"].str.capitalize()

print("Train size:", train_absa.shape)
print("Test size:", test_absa.shape)

Train size: (10331, 3)
Test size: (6600, 3)


In [6]:
label_map = {"Negative": 0, "Neutral": 1, "Positive": 2}

train_absa["label"] = train_absa["sentiment"].map(label_map)
test_absa["label"] = test_absa["sentiment"].map(label_map)

In [7]:
train_absa["text"] = train_absa.apply(
    lambda x: f"{x['sentence']} [SEP] aspect: {x['aspect']}", axis=1
)

test_absa["text"] = test_absa.apply(
    lambda x: f"{x['sentence']} [SEP] aspect: {x['aspect']}", axis=1
)

In [8]:
train_dataset = Dataset.from_pandas(train_absa[["text", "label"]])
test_dataset = Dataset.from_pandas(test_absa[["text", "label"]])

In [9]:
model_name = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/10331 [00:00<?, ? examples/s]

Map:   0%|          | 0/6600 [00:00<?, ? examples/s]

In [11]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    save_strategy="epoch"
)

In [12]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [13]:
trainer.train()

Step,Training Loss


Step,Training Loss
500,0.696040
1000,0.503607
1500,0.430956


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1938, training_loss=0.5031290595622501, metrics={'train_runtime': 2155.1395, 'train_samples_per_second': 14.381, 'train_steps_per_second': 0.899, 'total_flos': 5280590569222656.0, 'train_loss': 0.5031290595622501, 'epoch': 3.0})

In [18]:
results = trainer.predict(test_dataset)

from sklearn.metrics import accuracy_score, f1_score, classification_report

y_pred = results.predictions.argmax(axis=1)
y_true = results.label_ids

# Metrics
accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average="weighted")

print("ModernBERT (Fine-tuned) Results:")
print("----------------------------------")
print("Accuracy:", round(accuracy, 4))
print("F1 Score:", round(f1, 4))

# Convert labels back to text
labels_map_reverse = {0: "Negative", 1: "Neutral", 2: "Positive"}

y_pred_labels = [labels_map_reverse[p] for p in y_pred]
y_true_labels = [labels_map_reverse[t] for t in y_true]

print("\nClassification Report:\n")
print(classification_report(y_true_labels, y_pred_labels))

ModernBERT (Fine-tuned) Results:
----------------------------------
Accuracy: 0.8112
F1 Score: 0.8027

Classification Report:

              precision    recall  f1-score   support

    Negative       0.83      0.83      0.83      2246
     Neutral       0.55      0.38      0.45       845
    Positive       0.84      0.90      0.87      3509

    accuracy                           0.81      6600
   macro avg       0.74      0.70      0.72      6600
weighted avg       0.80      0.81      0.80      6600

